# Black-Litterman Model Backtest Toolkit

A backtesting toolkit: Input parameters → Run backtest → Return results

Core idea:
1. Define parameter acquisition functions (how to obtain μ and Σ)
2. Define viewpoint functions (how to obtain P, q, Ω)
3. Run backtest for comparison
4. View the improvement effect


In [28]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

## 1. Markowitz - Standard mean-variance optimization

# When creating a Markowitz object, you can modify these parameters:

mvo = Markowitz(
expected_returns = mu, # ← Modifiable: Expected return vector

cov_matrix = sigma, # ← Modifiable: Covariance matrix

risk_aversion = 100 # ← Modifiable: Risk aversion coefficient λ (default 100)


# During optimization, you can modify these parameters:

weights = mvo.optimize(
allow_shorting = False # ← Modifiable: Whether short selling is allowed

In [33]:
class Markowitz:
    
    def __init__(self, expected_returns: np.ndarray, cov_matrix: np.ndarray, 
                 risk_aversion: float = 100):
        """
        Parameters:
        - expected_returns:  (n_assets,)
        - cov_matrix:  (n_assets, n_assets)
        - risk_aversion:  λ
        """
        self.mu = expected_returns
        self.sigma = cov_matrix
        self.lambda_aversion = risk_aversion
        self.n_assets = len(expected_returns)
    
    def optimize(self, allow_shorting: bool = False):
        """
        Returns:
        -  (n_assets,)
        """
        def objective(w):
            port_return = -np.dot(w, self.mu)
            port_variance = np.dot(w, np.dot(self.sigma, w))
            return port_return + self.lambda_aversion * port_variance / 2
        
        constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
        
        if allow_shorting:
            bounds = [(None, None)] * self.n_assets
        else:
            bounds = [(0, 1)] * self.n_assets
        
        x0 = np.array([1.0 / self.n_assets] * self.n_assets)
        result = minimize(objective, x0, method='SLSQP', bounds=bounds, 
                         constraints=constraints, options={'maxiter': 1000})
        
        return result.x

## 2. Black-Litterman - basic version

When creating a BlackLitterman object, you can modify these parameters:

```python
bl = BlackLitterman(
    market_weights = w_m,        # ← Modifiable: Market portfolio weights
    cov_matrix = sigma,          # ← Modifiable: Covariance matrix
    risk_aversion = 100,         # ← Modifiable: Risk aversion coefficient λ
    tau = 0.05                   # ← Modifiable: Prior uncertainty scaling (0.05 = trust market)
)

# When updating with views:
mu_post, sigma_post = bl.update_with_views(
    P = P_matrix,                # ← Modifiable: View matrix (n_views, n_assets)
    q = q_vector,                # ← Modifiable: View returns (n_views,)
    omega = omega_matrix         # ← Modifiable: View uncertainty (None = auto-calculate)
)

# Getting optimized portfolio:
weights, mu_post, sigma_post = bl.optimize_with_views(
    P = P_matrix,
    q = q_vector,
    omega = omega_matrix,
    allow_shorting = False       # ← Modifiable: Whether to allow short selling
)
```

In [34]:
class BlackLitterman:
    """Black-Litterman """
    
    def __init__(self, market_weights: np.ndarray, cov_matrix: np.ndarray,
                 risk_aversion: float = 100, tau: float = 0.05):
        """
        Parameters:
        - market_weights:  (n_assets,)
        - cov_matrix:  (n_assets, n_assets)
        - risk_aversion: 
        - tau: 
        """
        self.w_market = market_weights
        self.sigma = cov_matrix
        self.lambda_aversion = risk_aversion
        self.tau = tau
        self.n_assets = len(market_weights)
        
        # imply return
        self.pi = self.lambda_aversion * np.dot(self.sigma, self.w_market)
    
    def update_with_views(self, P: np.ndarray, q: np.ndarray, 
                         omega: np.ndarray = None):
        """
        
        
        Parameters:
        - P:  (n_views, n_assets)
        - q:  (n_views,)
        - omega: 
        
        Returns:
        - mu_post: 
        - sigma_post: 
        """
        P = np.atleast_2d(P)
        q = np.atleast_1d(q)
        
    
        prior_cov = self.tau * self.sigma
        
        # automatically set omega if not provided
        if omega is None:
            view_variance = np.array([np.dot(P[i], np.dot(self.sigma, P[i])) 
                                     for i in range(len(P))])
            omega = np.diag(view_variance * self.tau)
        
        # compute posterior parameters
        omega_inv = np.linalg.inv(omega)
        sigma_prior_inv = np.linalg.inv(prior_cov)
        
        sigma_post_inv = sigma_prior_inv + np.dot(P.T, np.dot(omega_inv, P))
        sigma_post = np.linalg.inv(sigma_post_inv)
        
        term1 = np.dot(sigma_prior_inv, self.pi)
        term2 = np.dot(P.T, np.dot(omega_inv, q))
        mu_post = np.dot(sigma_post, term1 + term2)
        
        return mu_post, sigma_post
    
    def optimize_with_views(self, P: np.ndarray, q: np.ndarray,
                           omega: np.ndarray = None,
                           allow_shorting: bool = False):
        """
        
        Returns:
        - weights: 
        - mu_post: 
        - sigma_post: 
        """
        mu_post, sigma_post = self.update_with_views(P, q, omega)
        mvo = Markowitz(mu_post, sigma_post, self.lambda_aversion)
        weights = mvo.optimize(allow_shorting)
        
        return weights, mu_post, sigma_post

## 3. Rolling Window Backtest

When creating a RollingWindowBacktest object, you can modify these parameters:

```python
backtest = RollingWindowBacktest(
    returns_df = returns,           # ← Modifiable: Returns DataFrame (dates × assets)
    train_window = 252*5,           # ← Modifiable: Training window size in days (default: 5 years)
    rebalance_freq = 252            # ← Modifiable: Rebalancing frequency in days (default: 1 year)
)

# When running backtest_bl:
portfolio_values, portfolio_returns, weights = backtest.backtest_bl(
    get_params_func = get_standard_params,      # ← Modifiable: Function to get (mu, sigma)
    get_views_func = get_standard_views,        # ← Modifiable: Function to get (P, q, omega)
    market_weights = None,                      # ← Modifiable: Market weights (None = equal-weighted)
    tau = 0.05                                  # ← Modifiable: Prior uncertainty scaling (0.05 = trust market)
)
```

In [35]:
class RollingWindowBacktest:
    """Rolling window backtest"""
    
    def __init__(self, returns_df: pd.DataFrame, train_window: int = 252*5,
                 rebalance_freq: int = 252):
        """
        Parameters:
        - returns_df: returns DataFrame
        - train_window: training window size (days)
        - rebalance_freq: rebalancing frequency (days)
        """
        self.returns = returns_df
        self.train_window = train_window
        self.rebalance_freq = rebalance_freq
        self.n_assets = len(returns_df.columns)
    
    def backtest_markowitz(self, get_params_func):
        """
        Markowitz backtest
        
        Parameters:
        - get_params_func: function that takes training data and returns (mu, sigma)
        
        Returns:
        - portfolio_values: portfolio value time series
        - portfolio_returns: portfolio returns time series
        - weights_history: portfolio weights history
        """
        portfolio_values = [1.0]
        weights_history = []
        portfolio_returns = []
        
        for t in range(self.train_window, len(self.returns), self.rebalance_freq):
            train_data = self.returns.iloc[t-self.train_window:t]
            test_start = t
            test_end = min(t + self.rebalance_freq, len(self.returns))
            
            mu, sigma = get_params_func(train_data)
            
            mvo = Markowitz(mu, sigma)
            weights = mvo.optimize()
            weights_history.append(weights)
            
            test_data = self.returns.iloc[test_start:test_end]
            for ret_vec in test_data.values:
                port_ret = np.dot(weights, ret_vec)
                portfolio_returns.append(port_ret)
                portfolio_values.append(portfolio_values[-1] * (1 + port_ret))
        
        return (np.array(portfolio_values), np.array(portfolio_returns), 
                weights_history)
    
    def backtest_bl(self, get_params_func, get_views_func, 
                   market_weights=None, tau=0.05):
        """
        Black-Litterman backtest
        
        Parameters:
        - get_params_func: function that takes training data and returns (mu, sigma)
        - get_views_func: function that takes training data and returns (P, q, omega)
        - market_weights: market portfolio weights, None means equal-weighted
        - tau: prior uncertainty scaling factor
        
        Returns:
        - portfolio_values, portfolio_returns, weights_history
        """
        if market_weights is None:
            market_weights = np.array([1.0 / self.n_assets] * self.n_assets)
        
        portfolio_values = [1.0]
        weights_history = []
        portfolio_returns = []
        
        for t in range(self.train_window, len(self.returns), self.rebalance_freq):
            train_data = self.returns.iloc[t-self.train_window:t]
            test_start = t
            test_end = min(t + self.rebalance_freq, len(self.returns))
            
            _, sigma = get_params_func(train_data)
            P, q, omega = get_views_func(train_data)
            
            bl = BlackLitterman(market_weights, sigma, tau=tau)
            weights, _, _ = bl.optimize_with_views(P, q, omega)
            weights_history.append(weights)
            
            test_data = self.returns.iloc[test_start:test_end]
            for ret_vec in test_data.values:
                port_ret = np.dot(weights, ret_vec)
                portfolio_returns.append(port_ret)
                portfolio_values.append(portfolio_values[-1] * (1 + port_ret))
        
        return (np.array(portfolio_values), np.array(portfolio_returns),
                weights_history)

## 4. Performance Metrics

When creating a PerformanceMetrics object, you can modify these parameters:

```python
metrics = PerformanceMetrics(
    portfolio_values = portfolio_values,     # ← Modifiable: Portfolio value time series (from backtest)
    returns = portfolio_returns,             # ← Modifiable: Daily returns series (from backtest)
    rf_rate = 0.02,                         # ← Modifiable: Annual risk-free rate (default: 0.02 = 2%)
    weights_history = weights                # ← Modifiable: Portfolio weights history (optional, for turnover)
)

# Get all metrics at once:
all_metrics = metrics.get_metrics()  # Returns dict with 6 metrics:
                                     # Annual Return, Annual Volatility, Sharpe Ratio, 
                                     # Max Drawdown, Turnover, VaR(95%)

# Or calculate individual metrics:
annual_ret = metrics.annual_return()           # Annualized return
annual_vol = metrics.annual_volatility()       # Annualized volatility
sharpe = metrics.sharpe_ratio()                # Sharpe ratio
max_dd = metrics.max_drawdown()                # Maximum drawdown
turnover = metrics.turnover()                  # Average turnover (portfolio rebalancing cost)
var95 = metrics.var_95()                       # Value at Risk at 95% confidence level

# Compare two strategies:
comparison = compare_results(
    baseline_results = {'values': bl_values, 'returns': bl_returns, 'weights': bl_weights},
    experiment_results = {'values': exp_values, 'returns': exp_returns, 'weights': exp_weights}
)
# Returns DataFrame with: Baseline, Experiment, and Improvement columns

# Compare three strategies (Markowitz, Standard BL, Improved BL):
comparison = compare_three_strategies(
    markowitz_results = {'values': m_values, 'returns': m_returns, 'weights': m_weights},
    baseline_results = {'values': bl_values, 'returns': bl_returns, 'weights': bl_weights},
    experiment_results = {'values': exp_values, 'returns': exp_returns, 'weights': exp_weights}
)
# Returns DataFrame with all three strategies and their metrics

# Plot cumulative returns for all three strategies:
plot_cumulative_returns(
    markowitz_results = {'values': m_values, 'returns': m_returns, ...},
    baseline_results = {'values': bl_values, 'returns': bl_returns, ...},
    experiment_results = {'values': exp_values, 'returns': exp_returns, ...},
    title = 'Cumulative Returns Comparison'
)
```

In [29]:
class PerformanceMetrics:
    """Performance metrics calculation"""
    
    def __init__(self, portfolio_values: np.ndarray, returns: np.ndarray,
                 rf_rate: float = 0.02, weights_history: list = None):
        """
        Parameters:
        - portfolio_values: portfolio value time series
        - returns: daily returns time series
        - rf_rate: annual risk-free rate
        - weights_history: portfolio weights history (for turnover calculation)
        """
        self.values = portfolio_values
        self.returns = returns
        self.rf = rf_rate
        self.weights_history = weights_history
    
    def annual_return(self):
        """Annualized return"""
        total_return = (self.values[-1] / self.values[0]) - 1
        years = len(self.returns) / 252
        return (1 + total_return) ** (1 / years) - 1
    
    def annual_volatility(self):
        """Annualized volatility"""
        return np.std(self.returns) * np.sqrt(252)
    
    def sharpe_ratio(self):
        """Sharpe ratio"""
        annual_ret = self.annual_return()
        annual_vol = self.annual_volatility()
        if annual_vol == 0:
            return 0
        return (annual_ret - self.rf) / annual_vol
    
    def max_drawdown(self):
        """Maximum drawdown"""
        cum_max = np.maximum.accumulate(self.values)
        drawdown = (self.values - cum_max) / cum_max
        return np.min(drawdown)
    
    def turnover(self):
        """Average annual turnover"""
        if self.weights_history is None or len(self.weights_history) < 2:
            return 0
        
        turnovers = []
        for i in range(1, len(self.weights_history)):
            w_old = self.weights_history[i-1]
            w_new = self.weights_history[i]
            turnover_i = np.sum(np.abs(w_new - w_old)) / 2
            turnovers.append(turnover_i)
        
        avg_turnover = np.mean(turnovers)
        return avg_turnover
    
    def var_95(self):
        """Value at Risk at 95% confidence level"""
        return np.percentile(self.returns, 5)
    
    def get_metrics(self):
        """Return all metrics"""
        return {
            'Annual Return': self.annual_return(),
            'Annual Volatility': self.annual_volatility(),
            'Sharpe Ratio': self.sharpe_ratio(),
            'Max Drawdown': self.max_drawdown(),
            'Turnover': self.turnover(),
            'VaR(95%)': self.var_95(),
        }


def compare_results(baseline_results: dict, experiment_results: dict):
    """Compare baseline and experiment results"""
    baseline_metrics = PerformanceMetrics(
        baseline_results['values'], 
        baseline_results['returns'],
        weights_history=baseline_results.get('weights')
    ).get_metrics()
    
    experiment_metrics = PerformanceMetrics(
        experiment_results['values'],
        experiment_results['returns'],
        weights_history=experiment_results.get('weights')
    ).get_metrics()
    
    comparison = pd.DataFrame({
        'Baseline (Standard BL)': baseline_metrics,
        'Experiment (Modified BL)': experiment_metrics,
        'Improvement': {k: experiment_metrics[k] - baseline_metrics[k] 
                       for k in baseline_metrics.keys()}
    })
    
    return comparison


def compare_three_strategies(markowitz_results: dict, baseline_results: dict, 
                            experiment_results: dict):
    """Compare three strategies: Markowitz, Standard BL, Improved BL"""
    markowitz_metrics = PerformanceMetrics(
        markowitz_results['values'],
        markowitz_results['returns'],
        weights_history=markowitz_results.get('weights')
    ).get_metrics()
    
    baseline_metrics = PerformanceMetrics(
        baseline_results['values'],
        baseline_results['returns'],
        weights_history=baseline_results.get('weights')
    ).get_metrics()
    
    experiment_metrics = PerformanceMetrics(
        experiment_results['values'],
        experiment_results['returns'],
        weights_history=experiment_results.get('weights')
    ).get_metrics()
    
    comparison = pd.DataFrame({
        'Markowitz': markowitz_metrics,
        'Standard BL': baseline_metrics,
        'Improved BL': experiment_metrics,
    })
    
    return comparison


def plot_cumulative_returns(markowitz_results: dict, baseline_results: dict,
                           experiment_results: dict, title: str = 'Cumulative Returns Comparison'):
    """
    Plot cumulative returns for three strategies
    
    Parameters:
    - markowitz_results: {'values': array, 'returns': array, ...}
    - baseline_results: {'values': array, 'returns': array, ...}
    - experiment_results: {'values': array, 'returns': array, ...}
    - title: plot title
    """
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(14, 7))
    
    # Plot cumulative returns
    plt.plot(markowitz_results['values'], label='Markowitz', linewidth=2, alpha=0.8)
    plt.plot(baseline_results['values'], label='Standard BL', linewidth=2, alpha=0.8)
    plt.plot(experiment_results['values'], label='Improved BL', linewidth=2, alpha=0.8)
    
    plt.xlabel('Trading Days', fontsize=12)
    plt.ylabel('Cumulative Return (Portfolio Value)', fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.legend(fontsize=11, loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Parameter Acquisition Functions (customize as needed)

For your BL improvement experiments, you mainly need to modify these parameters:

```python
# YOUR KEY PARAMETERS TO MODIFY:

# 1. Improved Covariance Matrix (Σ)
# Pass via: custom_sigma = your_improved_covariance_matrix
def get_custom_params(train_returns, custom_sigma=None):
    mu = train_returns.mean().values * 252
    if custom_sigma is not None:
        sigma = custom_sigma  # ← MODIFIABLE: Your improved Σ
    else:
        sigma = train_returns.cov().values  # Default: sample covariance
    return mu, sigma

# 2. Improved View Uncertainty (Ω)
# Pass via: custom_omega = your_improved_uncertainty_matrix
def get_custom_views(train_returns, custom_P=None, custom_q=None, 
                    custom_omega=None):
    # Views P, q use standard momentum signal (FIXED)
    if custom_P is not None and custom_q is not None:
        P, q = custom_P, custom_q  # Optional: custom views
    else:
        P, q, _ = get_standard_views(train_returns)  # Standard: momentum-based
    
    # View uncertainty can be improved independently
    omega = custom_omega  # ← MODIFIABLE: Your improved Ω (or None = auto-calculate)
    
    return P, q, omega
```

In [36]:
# Baseline: standard parameters and views
def get_standard_params(train_returns):
    """Standard parameters: based on sample statistics"""
    mu = train_returns.mean().values * 252  # annualized
    sigma = train_returns.cov().values
    return mu, sigma


def get_standard_views(train_returns):
    """Standard views: based on momentum signal"""
    n_assets = len(train_returns.columns)
    recent_returns = train_returns.tail(20).mean() * 252
    top_indices = np.argsort(recent_returns.values)[-2:]
    
    P = np.zeros((2, n_assets))
    for i, idx in enumerate(top_indices):
        P[i, idx] = 1.0
    
    q = np.array([0.05, 0.05])
    omega = None  # auto-calculate
    
    return P, q, omega


# Custom parameters and views (for improvement experiments)
def get_custom_params(train_returns, custom_sigma=None):
    """
    Custom parameters
    
    If you have an improved covariance matrix, pass it as custom_sigma
    """
    mu = train_returns.mean().values * 252
    if custom_sigma is not None:
        sigma = custom_sigma
    else:
        sigma = train_returns.cov().values
    return mu, sigma


def get_custom_views(train_returns, custom_P=None, custom_q=None, 
                    custom_omega=None):
    """
    Custom views
    
    If you have improved views, pass custom_P, custom_q, custom_omega
    If only improved uncertainty (omega), pass custom_omega and leave P, q as None
    """
    if custom_P is not None and custom_q is not None:
        P = custom_P
        q = custom_q
    else:
        # Use standard momentum-based views for P and q
        P, q, _ = get_standard_views(train_returns)
    
    # Handle omega separately - can be improved independently
    omega = custom_omega
    
    return P, q, omega

## 6. Run Experiment Framework (Three-Strategy Comparison)

Complete backtesting with automatic comparison of Markowitz, Standard BL, and Improved BL

In [38]:
def run_experiment(returns_data, custom_sigma=None, custom_P=None, 
                  custom_q=None, custom_omega=None, tau=0.05,
                  train_window=252*5, rebalance_freq=252, plot=True):
    """
    Run comparison experiment with Markowitz, Standard BL, and Improved BL
    
    Parameters:
    - returns_data: returns DataFrame (dates × assets)
    - custom_sigma: your improved covariance matrix (optional)
    - custom_P: your improved view matrix (optional)
    - custom_q: your improved view returns (optional)
    - custom_omega: your improved uncertainty matrix (optional)
    - tau: prior uncertainty scaling factor
    - train_window: training window size (days)
    - rebalance_freq: rebalancing frequency (days)
    - plot: whether to plot cumulative returns
    
    Returns:
    - comparison results DataFrame
    - results dictionary with all three strategies
    """
    
    backtest = RollingWindowBacktest(
        returns_data,
        train_window=train_window,
        rebalance_freq=rebalance_freq
    )
    
    print("="*80)
    print("BACKTEST RUNNING...")
    print("="*80)
    
    # Strategy 1: Markowitz (Mean-Variance Optimization)
    print("→ Strategy 1: Markowitz (Mean-Variance Optimization)...")
    markowitz_values, markowitz_returns, markowitz_weights = backtest.backtest_markowitz(
        get_params_func=get_standard_params
    )
    markowitz = {'values': markowitz_values, 'returns': markowitz_returns, 'weights': markowitz_weights}
    
    # Strategy 2: Standard BL
    print("→ Strategy 2: Standard BL...")
    baseline_values, baseline_returns, baseline_weights = backtest.backtest_bl(
        get_params_func=get_standard_params,
        get_views_func=get_standard_views,
        tau=tau
    )
    baseline = {'values': baseline_values, 'returns': baseline_returns, 'weights': baseline_weights}
    
    # Strategy 3: Improved BL
    print("→ Strategy 3: Improved BL...")
    
    def custom_params_wrapper(train_data):
        return get_custom_params(train_data, custom_sigma=custom_sigma)
    
    def custom_views_wrapper(train_data):
        return get_custom_views(train_data, custom_P=custom_P, 
                              custom_q=custom_q, custom_omega=custom_omega)
    
    experiment_values, experiment_returns, experiment_weights = backtest.backtest_bl(
        get_params_func=custom_params_wrapper,
        get_views_func=custom_views_wrapper,
        tau=tau
    )
    experiment = {'values': experiment_values, 'returns': experiment_returns, 'weights': experiment_weights}
    
    # Comparison results
    print("="*80)
    comparison = compare_three_strategies(markowitz, baseline, experiment)
    print("\nCOMPARISON RESULTS:")
    print(comparison.round(4).to_string())
    print("="*80)
    
    # Plot cumulative returns
    if plot:
        plot_cumulative_returns(markowitz, baseline, experiment)
    
    results = {
        'markowitz': markowitz,
        'standard_bl': baseline,
        'improved_bl': experiment
    }
    
    return comparison, results


def run_experiment_improve_sigma_only(returns_data, improved_sigma, tau=0.05,
                                     train_window=252*5, rebalance_freq=252, plot=True):
    """
    Test ONLY improved Σ (covariance matrix)
    Other parameters use standard BL settings
    
    Parameters:
    - returns_data: returns DataFrame
    - improved_sigma: your improved covariance matrix (required)
    - tau: prior uncertainty scaling
    - train_window: training window size (days)
    - rebalance_freq: rebalancing frequency (days)
    - plot: whether to plot cumulative returns
    
    Returns:
    - comparison results DataFrame
    - results dictionary
    """
    print("\n" + "="*80)
    print("TEST: Improved Σ (Covariance) ONLY")
    print("P, q, Ω use standard BL settings")
    print("="*80 + "\n")
    
    return run_experiment(
        returns_data,
        custom_sigma=improved_sigma,
        custom_P=None,              # Use standard P
        custom_q=None,              # Use standard q
        custom_omega=None,          # Use standard omega (auto-calculate)
        tau=tau,
        train_window=train_window,
        rebalance_freq=rebalance_freq,
        plot=plot
    )


def run_experiment_improve_omega_only(returns_data, improved_omega, tau=0.05,
                                     train_window=252*5, rebalance_freq=252, plot=True):
    """
    Test ONLY improved Ω (view uncertainty matrix)
    Other parameters use standard BL settings
    
    Parameters:
    - returns_data: returns DataFrame
    - improved_omega: your improved uncertainty matrix (required)
    - tau: prior uncertainty scaling
    - train_window: training window size (days)
    - rebalance_freq: rebalancing frequency (days)
    - plot: whether to plot cumulative returns
    
    Returns:
    - comparison results DataFrame
    - results dictionary
    """
    print("\n" + "="*80)
    print("TEST: Improved Ω (View Uncertainty) ONLY")
    print("Σ, P, q use standard BL settings")
    print("="*80 + "\n")
    
    return run_experiment(
        returns_data,
        custom_sigma=None,          # Use standard covariance (sample)
        custom_P=None,              # Use standard P
        custom_q=None,              # Use standard q
        custom_omega=improved_omega, # Use your improved omega
        tau=tau,
        train_window=train_window,
        rebalance_freq=rebalance_freq,
        plot=plot
    )